### Sensitivity Kernel

**Remarks:** the model is extended below the inverted 120 m grid (constant half-space Vs) to `MAX_MODEL_DEPTH = 300 m` before calling disba. Otherwise the 2 Hz sensitivity (which reaches ~110 m+) collapses onto a spike at the 120 m floor instead of forming a real lobe. The plot now extends to 160 m so we can see where each frequency truly peaks.


In [ ]:
# Import necessary dependencies
import sys
import pickle
import numpy as np

import matplotlib.pyplot as plt
from disba import PhaseSensitivity

sys.path.append('..')
from src.inv import animate_sensitivity_kernels, save_sensitivity_kernel_plots

#### 1. Load Inversion Results

In [ ]:
# Load the saved 2D matrix and metadata
saved_data = np.load("../results/inv_outputs_urban/final_2D_matrix.npz")
vs_2d_matrix_mean = saved_data["vs_matrix_mean"]
vs_2d_matrix_best = saved_data["vs_matrix_best"]
positions = saved_data["positions"]
z_grid = saved_data["z_grid"]

# Load the entire list of InversionResults objects back into memory
with open("../results/inv_outputs_urban/all_results_list.pkl", "rb") as f:
    all_results = pickle.load(f)

In [ ]:
# Pick a column in the middle of 2D profile to test
middle_idx = len(positions) // 2
position_m = positions[middle_idx]
print(f"Testing sensitivity for position: {position_m:.2f} m")

# Extract the 1D shear wave velocity profile (in m/s)
vs_1d_ms = vs_2d_matrix_best[:, middle_idx]

#### 2. Convert to Disba's format (Thickness, Vp, Vs, Rho in km and km/s)

In [ ]:
# Build the 1D model for the kernel.
# KEY: extend the profile BELOW the inverted grid (hold the deepest /
# half-space Vs constant) down to MAX_MODEL_DEPTH. The inverted model bottoms at
# 120 m, but the 2 Hz wave senses to ~110 m+, so its sensitivity gets absorbed by
# the terminal half-space and shows up as a SPIKE at 120 m. Extending deeper lets
# the low-frequency kernels form a real lobe, so we can see where they truly peak.
MAX_MODEL_DEPTH = 300.0  # m, well below the deepest 2 Hz sensitivity

z_depths = np.abs(z_grid)
dz = float(np.median(np.diff(z_depths)))
z_ext = np.arange(z_depths.max() + dz, MAX_MODEL_DEPTH + dz, dz)
z_model = np.concatenate([z_depths, z_ext])

# Extend Vs downward with the deepest (half-space) value.
vs_1d_ext = np.concatenate([vs_1d_ms, np.full(z_ext.size, vs_1d_ms[-1])])

# Disba model: thickness (km), Vp, Vs, density. Last entry = half-space.
thickness_m = np.append(np.diff(z_model), 10.0)
thickness_km = thickness_m / 1000.0
vs_kms = vs_1d_ext / 1000.0
vp_kms = vs_kms * 2.0
rho_gcm3 = np.full_like(vs_kms, 1.0)  # matches inversion assumption

velocity_model = np.column_stack((thickness_km, vp_kms, vs_kms, rho_gcm3))

#### 3. Calculate the sensitivity kernels

In [ ]:
# Initialize the calculator with our specified velocity model
ps = PhaseSensitivity(*velocity_model.T)

# Pick the frequencies we want to test (from dispersion images)
test_frequencies = [2.0, 3.0, 4.0, 5.0, 6.0]

In [ ]:
plt.figure(figsize=(5, 7))

for f in test_frequencies:
    period = 1.0 / f
    k = ps(period, mode=0, wave="rayleigh", parameter="velocity_s")
    depth_m = k.depth * 1000.0
    plt.plot(k.kernel, depth_m, label=f"{f} Hz", linewidth=2)

# Wavelength-based resolution guides (lambda_max ~ 222 m from the dispersion):
# well-resolved core ~ lambda_max/3 ~ 75 m ; optimistic edge ~ lambda_max/2 ~ 110 m
plt.axhline(75,  color="0.5", ls=":",  lw=1.2)
plt.axhline(110, color="0.5", ls="--", lw=1.2)
# plt.text(0.0, 75,  " ~lambda/3 (well resolved)", va="bottom", fontsize=8, color="0.4")
# plt.text(0.0, 110, " ~lambda/2 (max usable)",    va="bottom", fontsize=8, color="0.4")

plt.ylim(160, 0)   # show below 120 m so the low-frequency lobe is visible
plt.xlim(left=0)

plt.xlabel(r"Sensitivity Kernel ($\partial c / \partial V_s$)", fontsize=14)
plt.ylabel("Depth (m)", fontsize=14)
plt.title(f"Rayleigh Wave Sensitivity (model extended to {MAX_MODEL_DEPTH:.0f} m)\n"
          f"at Position = {position_m:.1f} m", fontsize=14)

plt.grid(True, linestyle="--", alpha=0.7)
plt.legend(loc="lower right", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Animated kernels along the whole profile, using the same deeper model so the
# low-frequency lobes are not artifacted (max_model_depth extends below the grid).
animate_sensitivity_kernels(
    positions, vs_2d_matrix_best, z_grid,
    x_max=0.05, max_model_depth=300.0, ylim=(0, 160)
)

In [ ]:
save_sensitivity_kernel_plots(
    positions, 
    vs_2d_matrix_best, 
    z_grid, 
    test_frequencies=[2.0, 3.0, 4.0, 5.0], 
    x_max=0.05,
    max_model_depth=300.0,
    ylim=(0, 160),
    save_dir="../results/inv_sensitivity_urban"
)